# Intelligent Travel Assistant Chatbot (Fuzzy Logic)

A conversational travel recommender for India. It asks for your **budget (INR)**, preferred **climate (°C)** and **interest in adventure/heritage (0-100)**, then uses a **fuzzy-logic inference engine** (`scikit-fuzzy`) to rank 50 curated destinations. Live weather and 5-day forecasts come from **OpenWeatherMap**, and a session-based feedback loop (`simulate_backpropagation`) adjusts scores when you reject a suggestion. The UI is built with **Gradio**.

**How to run:** `Runtime -> Run all`. When asked, provide an OpenWeatherMap API key (free at <https://openweathermap.org/api>). Best practice: store it as a Colab Secret named `OPENWEATHER_API_KEY` (key icon in the left sidebar) so it is never saved in the notebook.

*VIT-AP University - B.Tech CSE, Final Project (CSE2009). Guide: Dr. Radha Mohan Pattanayak. Team: Rishitha Uppala, Jyothi Shravani Mamidi, Charishma Jagarapu, Charmesh Chejarla.*

## 1. Install dependencies

In [ ]:
!pip install -q "gradio>=5,<6" scikit-fuzzy requests

## 2. Imports and API key

In [ ]:
import os
import time
import datetime
from getpass import getpass
from urllib.parse import quote_plus

import numpy as np
import requests
import skfuzzy as fuzz
import gradio as gr

# OpenWeatherMap API key -- never hard-code it in the notebook.
# Priority: Colab Secrets (key icon in the left sidebar, name it OPENWEATHER_API_KEY)
#           -> environment variable OPENWEATHER_API_KEY -> interactive prompt.
API_KEY = None
try:
    from google.colab import userdata
    API_KEY = userdata.get("OPENWEATHER_API_KEY")
except Exception:
    pass
API_KEY = API_KEY or os.environ.get("OPENWEATHER_API_KEY") or getpass("Enter your OpenWeatherMap API key: ")


## 3. Dataset: 50 Indian destinations
Each destination has 0-100 scores for `budget`, `climate`, `interest`, a `weather_city` name for the API, and 3/5/7-day itineraries. In the fuzzy engine every destination acts as one implicit rule (e.g. *Manali: budget 40, climate 20, interest 85*).

In [ ]:
# Dataset: 50 Indian destinations.
# budget / climate / interest are scores on a 0-100 scale (higher budget = pricier, higher climate = hotter).
# Each destination doubles as an implicit fuzzy rule (see fuzzy_recommendation below).
base_destinations = {
'Manali': {'budget': 40, 'climate': 20, 'interest': 85, 'weather_city': 'Manali', 'itinerary': {
  '3-day': ['Day 1: Arrive and explore Mall Road.', 'Day 2: Visit Hadimba Temple and Old Manali.', 'Day 3: Day trip to Solang Valley for adventure sports.'],
  '5-day': ['Day 1: Arrive and explore Mall Road.', 'Day 2: Visit Hadimba Temple and Old Manali.', 'Day 3: Day trip to Solang Valley for adventure sports.', 'Day 4: Visit Vashisht Hot Springs and Jogini Falls.', 'Day 5: Explore Naggar Castle before departure.'],
  '7-day': ['Day 1: Arrive and explore Mall Road.', 'Day 2: Visit Hadimba Temple and Old Manali.', 'Day 3: Day trip to Solang Valley for adventure sports.', 'Day 4: Day trip to Rohtang Pass (seasonal).', 'Day 5: Visit Vashisht Hot Springs and Jogini Falls.', 'Day 6: Explore Naggar Castle and Art Gallery.', 'Day 7: Relaxing walk along the Beas River and depart.']}},
'Goa': {'budget': 70, 'climate': 90, 'interest': 80, 'weather_city': 'Goa', 'itinerary': {
  '3-day': ['Day 1: Arrive in North Goa, relax at Baga Beach.', 'Day 2: Explore Anjuna Flea Market and Fort Aguada.', "Day 3: Visit Old Goa's churches before departure."],
  '5-day': ['Day 1: Arrive in North Goa, relax at Baga Beach.', 'Day 2: Explore Anjuna Flea Market and Fort Aguada.', 'Day 3: Explore Panjim and the Latin Quarter (Fontainhas).', 'Day 4: Day trip to South Goa (Palolem Beach).', 'Day 5: Visit a spice plantation and depart.'],
  '7-day': ['Day 1: Arrive in North Goa, relax at Baga Beach.', 'Day 2: Explore Anjuna Flea Market and Fort Aguada.', 'Day 3: Water sports at Calangute.', 'Day 4: Explore Panjim and the Latin Quarter (Fontainhas).', 'Day 5: Day trip to South Goa (Palolem & Agonda).', 'Day 6: Visit Dudhsagar Falls (seasonal) or a spice plantation.', 'Day 7: Relax at a serene beach like Morjim and depart.']}},
'Leh': {'budget': 45, 'climate': 15, 'interest': 90, 'weather_city': 'Leh', 'itinerary': {
  '3-day': ['Day 1: Acclimatize. Visit Leh Palace and Shanti Stupa.', 'Day 2: Visit Thiksey Monastery and Shey Palace.', 'Day 3: Explore Leh market before departure.'],
  '5-day': ['Day 1: Acclimatize. Visit Leh Palace and Shanti Stupa.', 'Day 2: Visit Thiksey Monastery and Shey Palace.', 'Day 3: Day trip to Khardung La Pass.', 'Day 4: Visit Hemis Monastery.', 'Day 5: Rafting on the Zanskar river (seasonal) and depart.'],
  '7-day': ['Day 1: Acclimatize.', 'Day 2: Visit Leh Palace and Shanti Stupa.', 'Day 3: Day trip to Khardung La Pass.', 'Day 4: Travel to Pangong Lake for an overnight stay.', 'Day 5: Return from Pangong and rest.', 'Day 6: Visit Hemis & Thiksey Monasteries.', 'Day 7: Explore local markets and depart.']}},
'Udaipur': {'budget': 50, 'climate': 70, 'interest': 65, 'weather_city': 'Udaipur', 'itinerary': {
  '3-day': ['Day 1: Arrive, evening boat ride on Lake Pichola.', 'Day 2: Visit the City Palace and Jagdish Temple.', 'Day 3: Explore Saheliyon Ki Bari and depart.'],
  '5-day': ['Day 1: Arrive, evening boat ride on Lake Pichola.', 'Day 2: Visit the City Palace and Jagdish Temple.', 'Day 3: Visit Sajjangarh (Monsoon Palace) for sunset.', 'Day 4: Explore Bagore Ki Haveli and the old city.', 'Day 5: Day trip to Kumbhalgarh Fort.'],
  '7-day': ['Day 1: Arrive, evening boat ride on Lake Pichola.', 'Day 2: Visit the City Palace and Jagdish Temple.', 'Day 3: Visit Sajjangarh (Monsoon Palace) for sunset.', 'Day 4: Explore Bagore Ki Haveli and the old city.', 'Day 5: Day trip to Kumbhalgarh Fort.', 'Day 6: Day trip to Ranakpur Jain Temple.', 'Day 7: Relax and shop in the old city.']}},
'Darjeeling': {'budget': 30, 'climate': 30, 'interest': 60, 'weather_city': 'Darjeeling', 'itinerary': {
  '3-day': ['Day 1: Arrive, stroll on the Mall Road.', 'Day 2: Early morning trip to Tiger Hill for sunrise, visit Batasia Loop.', 'Day 3: Take the Toy Train ride, depart.'],
  '5-day': ['Day 1: Arrive, stroll on the Mall Road.', 'Day 2: Early morning trip to Tiger Hill for sunrise, visit Batasia Loop.', 'Day 3: Take the Toy Train ride, visit Himalayan Mountaineering Institute.', 'Day 4: Explore a tea estate.', 'Day 5: Visit the Peace Pagoda and depart.'],
  '7-day': ['Day 1: Arrive, stroll on the Mall Road.', 'Day 2: Early morning trip to Tiger Hill for sunrise, visit Batasia Loop.', 'Day 3: Take the Toy Train ride, visit Himalayan Mountaineering Institute.', 'Day 4: Explore a tea estate.', 'Day 5: Day trip to Mirik Lake.', 'Day 6: Visit Padmaja Naidu Himalayan Zoological Park.', "Day 7: Enjoy breakfast at Glenary's and depart."]}},
'Jaipur': {'budget': 50, 'climate': 80, 'interest': 70, 'weather_city': 'Jaipur', 'itinerary': {
  '3-day': ['Day 1: Visit City Palace and Jantar Mantar.', 'Day 2: Explore Amber Fort and Hawa Mahal.', 'Day 3: Shop at Johari Bazaar.'],
  '5-day': ['Day 1: Visit City Palace and Jantar Mantar.', 'Day 2: Explore Amber Fort.', 'Day 3: See Hawa Mahal & Jal Mahal.', 'Day 4: Visit Albert Hall Museum and Birla Mandir.', 'Day 5: Explore Nahargarh Fort for city views.'],
  '7-day': ['Day 1: Visit City Palace and Jantar Mantar.', 'Day 2: Explore Amber Fort.', 'Day 3: See Hawa Mahal & Jal Mahal.', 'Day 4: Visit Albert Hall Museum.', 'Day 5: Explore Nahargarh Fort.', 'Day 6: Day trip to Sanganer for block printing.', 'Day 7: Food tour in the old city.']}},
'Rishikesh': {'budget': 35, 'climate': 50, 'interest': 85, 'weather_city': 'Rishikesh', 'itinerary': {
  '3-day': ['Day 1: Arrive, visit Laxman Jhula and Ram Jhula.', 'Day 2: Experience river rafting.', 'Day 3: Attend Ganga Aarti at Triveni Ghat.'],
  '5-day': ['Day 1: Arrive, visit Laxman Jhula and Ram Jhula.', 'Day 2: Experience river rafting.', 'Day 3: Attend Ganga Aarti at Triveni Ghat.', 'Day 4: Visit the Beatles Ashram.', 'Day 5: Short trek to Neer Garh Waterfall.'],
  '7-day': ['Day 1: Arrive, visit Laxman Jhula and Ram Jhula.', 'Day 2: Experience river rafting.', 'Day 3: Attend Ganga Aarti at Triveni Ghat.', 'Day 4: Visit the Beatles Ashram.', 'Day 5: Short trek to Neer Garh Waterfall.', 'Day 6: Yoga and meditation session at an ashram.', 'Day 7: Explore local cafes.']}},
'Agra': {'budget': 40, 'climate': 75, 'interest': 60, 'weather_city': 'Agra', 'itinerary': {
  '3-day': ['Day 1: Visit the Taj Mahal at sunrise.', 'Day 2: Explore Agra Fort.', 'Day 3: Visit Fatehpur Sikri.'],
  '5-day': ['Day 1: Visit the Taj Mahal at sunrise.', 'Day 2: Explore Agra Fort.', 'Day 3: Visit Fatehpur Sikri.', 'Day 4: Visit Mehtab Bagh for sunset views of the Taj.', 'Day 5: Explore the Tomb of Itimad-ud-Daulah (Baby Taj).'],
  '7-day': ['Day 1: Visit the Taj Mahal at sunrise.', 'Day 2: Explore Agra Fort.', 'Day 3: Visit Fatehpur Sikri.', 'Day 4: Visit Mehtab Bagh for sunset views of the Taj.', 'Day 5: Explore the Tomb of Itimad-ud-Daulah (Baby Taj).', 'Day 6: Day trip to Mathura and Vrindavan.', 'Day 7: Explore local markets for marble crafts.']}},
'Ooty': {'budget': 45, 'climate': 40, 'interest': 65, 'weather_city': 'Ooty', 'itinerary': {
  '3-day': ['Day 1: Arrive, boat ride on Ooty Lake.', 'Day 2: Visit the Botanical Gardens.', 'Day 3: Ride the Nilgiri Mountain Railway (Toy Train).'],
  '5-day': ['Day 1: Arrive, boat ride on Ooty Lake.', 'Day 2: Visit the Botanical Gardens.', 'Day 3: Ride the Nilgiri Mountain Railway (Toy Train).', 'Day 4: Visit Doddabetta Peak for panoramic views.', 'Day 5: Explore a tea factory and museum.'],
  '7-day': ['Day 1: Arrive, boat ride on Ooty Lake.', 'Day 2: Visit the Botanical Gardens.', 'Day 3: Ride the Nilgiri Mountain Railway (Toy Train).', 'Day 4: Visit Doddabetta Peak for panoramic views.', 'Day 5: Explore a tea factory and museum.', 'Day 6: Day trip to Coonoor.', 'Day 7: Visit Pykara Lake and waterfalls.']}},
'Mysore': {'budget': 40, 'climate': 65, 'interest': 55, 'weather_city': 'Mysore', 'itinerary': {
  '3-day': ['Day 1: Visit the magnificent Mysore Palace.', 'Day 2: Explore Chamundi Hills and the Chamundeshwari Temple.', 'Day 3: Stroll through the Brindavan Gardens.'],
  '5-day': ['Day 1: Visit the magnificent Mysore Palace.', 'Day 2: Explore Chamundi Hills and the Chamundeshwari Temple.', 'Day 3: Stroll through the Brindavan Gardens.', "Day 4: Visit St. Philomena's Church and the Railway Museum.", 'Day 5: Day trip to Srirangapatna.'],
  '7-day': ['Day 1: Visit the magnificent Mysore Palace.', 'Day 2: Explore Chamundi Hills and the Chamundeshwari Temple.', 'Day 3: Stroll through the Brindavan Gardens.', "Day 4: Visit St. Philomena's Church and the Railway Museum.", 'Day 5: Day trip to Srirangapatna.', 'Day 6: Explore the Devaraja Market and try Mysore Pak.', 'Day 7: Visit the Karanji Lake and its aviary.']}},
'Kodaikanal': {'budget': 45, 'climate': 35, 'interest': 70, 'weather_city': 'Kodaikanal', 'itinerary': {
  '3-day': ['Day 1: Arrive, go boating on Kodai Lake.', "Day 2: Walk Coaker's Walk and visit Bryant Park.", 'Day 3: Explore Pillar Rocks and Guna Caves.'],
  '5-day': ['Day 1: Arrive, go boating on Kodai Lake.', "Day 2: Walk Coaker's Walk and visit Bryant Park.", 'Day 3: Explore Pillar Rocks and Guna Caves.', 'Day 4: Visit Berijam Lake (permit required).', "Day 5: Trek to Dolphin's Nose viewpoint."],
  '7-day': ['Day 1: Arrive, go boating on Kodai Lake.', "Day 2: Walk Coaker's Walk and visit Bryant Park.", 'Day 3: Explore Pillar Rocks and Guna Caves.', 'Day 4: Visit Berijam Lake (permit required).', "Day 5: Trek to Dolphin's Nose viewpoint.", 'Day 6: Visit the Silver Cascade Falls.', 'Day 7: Relax and explore local chocolate shops.']}},
'Hampi': {'budget': 30, 'climate': 70, 'interest': 75, 'weather_city': 'Hampi', 'itinerary': {
  '3-day': ['Day 1: Explore the Virupaksha Temple and Hampi Bazaar.', 'Day 2: Visit the Vittala Temple Complex (Stone Chariot).', "Day 3: Cross the river to the 'hippie island' of Anegundi."],
  '5-day': ['Day 1: Explore the Virupaksha Temple and Hampi Bazaar.', 'Day 2: Visit the Vittala Temple Complex (Stone Chariot).', "Day 3: Cross the river to the 'hippie island' of Anegundi.", "Day 4: Explore the Royal Enclosure and Queen's Bath.", 'Day 5: Hike Matanga Hill for sunrise/sunset views.'],
  '7-day': ['Day 1: Explore the Virupaksha Temple and Hampi Bazaar.', 'Day 2: Visit the Vittala Temple Complex (Stone Chariot).', "Day 3: Cross the river to the 'hippie island' of Anegundi.", "Day 4: Explore the Royal Enclosure and Queen's Bath.", 'Day 5: Hike Matanga Hill for sunrise/sunset views.', 'Day 6: Coracle ride on the Tungabhadra River.', 'Day 7: Explore lesser-known ruins and relax.']}},
'Alleppey': {'budget': 40, 'climate': 85, 'interest': 70, 'weather_city': 'Alappuzha', 'itinerary': {
  '3-day': ['Day 1: Board a houseboat for an overnight backwater cruise.', 'Day 2: Disembark and explore Alleppey Beach.', 'Day 3: Visit a coir-making unit before departure.'],
  '5-day': ['Day 1: Board a houseboat for an overnight backwater cruise.', 'Day 2: Disembark and check into a resort. Relax or try a canoe tour.', 'Day 3: Explore Alleppey Beach.', 'Day 4: Take a canoe tour through narrower canals.', 'Day 5: Visit Marari Beach.'],
  '7-day': ['Day 1: Board a houseboat for an overnight backwater cruise.', 'Day 2: Disembark and check into a resort.', 'Day 3: Explore Alleppey Beach.', 'Day 4: Take a canoe tour through narrower canals.', 'Day 5: Visit Marari Beach.', 'Day 6: Experience village life in Kuttanad.', 'Day 7: Enjoy an Ayurvedic massage.']}},
'Kolkata': {'budget': 50, 'climate': 85, 'interest': 60, 'weather_city': 'Kolkata', 'itinerary': {
  '3-day': ['Day 1: Visit the Victoria Memorial.', 'Day 2: Explore Howrah Bridge and the flower market.', 'Day 3: Stroll through Park Street and try local street food.'],
  '5-day': ['Day 1: Visit the Victoria Memorial.', 'Day 2: Explore Howrah Bridge and the flower market.', 'Day 3: Stroll through Park Street and try local street food.', "Day 4: Visit the Indian Museum and Kumartuli (potter's colony).", 'Day 5: Take a tram ride and explore College Street (book market).'],
  '7-day': ['Day 1: Visit the Victoria Memorial.', 'Day 2: Explore Howrah Bridge and the flower market.', 'Day 3: Stroll through Park Street.', "Day 4: Visit the Indian Museum and Kumartuli (potter's colony).", 'Day 5: Take a tram ride and explore College Street (book market).', 'Day 6: Day trip to the Sunderbans (mangrove forest).', 'Day 7: Explore the Dakshineswar Kali Temple.']}},
'Pondicherry': {'budget': 40, 'climate': 80, 'interest': 65, 'weather_city': 'Puducherry', 'itinerary': {
  '3-day': ['Day 1: Explore the French Quarter (White Town).', 'Day 2: Visit Auroville and the Matrimandir.', 'Day 3: Relax at Promenade Beach.'],
  '5-day': ['Day 1: Explore the French Quarter (White Town).', 'Day 2: Visit Auroville and the Matrimandir.', 'Day 3: Relax at Promenade Beach.', 'Day 4: Visit the Arikamedu archaeological site.', 'Day 5: Explore Paradise Beach.'],
  '7-day': ['Day 1: Explore the French Quarter (White Town).', 'Day 2: Visit Auroville and the Matrimandir.', 'Day 3: Relax at Promenade Beach.', 'Day 4: Visit the Arikamedu archaeological site.', 'Day 5: Explore Paradise Beach.', 'Day 6: Cafe hopping in the French Quarter.', 'Day 7: Visit the Botanical Gardens.']}},
'Shillong': {'budget': 30, 'climate': 45, 'interest': 60, 'weather_city': 'Shillong', 'itinerary': {
  '3-day': ['Day 1: Arrive, boat on Umiam Lake.', 'Day 2: Day trip to Cherrapunji (Nohkalikai Falls).', 'Day 3: Explore Shillong Peak and Elephant Falls.'],
  '5-day': ['Day 1: Arrive, boat on Umiam Lake.', 'Day 2: Day trip to Cherrapunji (Nohkalikai Falls).', 'Day 3: Explore Shillong Peak and Elephant Falls.', "Day 4: Day trip to Mawlynnong (Asia's cleanest village) and the Living Root Bridge.", 'Day 5: Explore the Don Bosco Museum.'],
  '7-day': ['Day 1: Arrive, boat on Umiam Lake.', 'Day 2: Day trip to Cherrapunji.', 'Day 3: Explore Shillong Peak and Elephant Falls.', 'Day 4: Day trip to Mawlynnong and the Living Root Bridge.', 'Day 5: Explore the Don Bosco Museum.', 'Day 6: Day trip to Dawki (clear river).', 'Day 7: Relax and shop at Police Bazaar.']}},
'Guwahati': {'budget': 35, 'climate': 65, 'interest': 55, 'weather_city': 'Guwahati', 'itinerary': {
  '3-day': ['Day 1: Visit the Kamakhya Temple.', 'Day 2: Cruise on the Brahmaputra River.', 'Day 3: Visit the Assam State Museum.'],
  '5-day': ['Day 1: Visit the Kamakhya Temple.', 'Day 2: Cruise on the Brahmaputra River.', 'Day 3: Visit the Assam State Museum.', 'Day 4: Visit the Pobitora Wildlife Sanctuary for rhinos.', 'Day 5: Explore the Umananda Temple on Peacock Island.'],
  '7-day': ['Day 1: Visit the Kamakhya Temple.', 'Day 2: Cruise on the Brahmaputra River.', 'Day 3: Visit the Assam State Museum.', 'Day 4: Visit the Pobitora Wildlife Sanctuary for rhinos.', 'Day 5: Explore the Umananda Temple on Peacock Island.', 'Day 6: Day trip to Hajo.', 'Day 7: Relax and try Assamese cuisine.']}},
'Tawang': {'budget': 30, 'climate': 15, 'interest': 85, 'weather_city': 'Tawang', 'itinerary': {
  '3-day': ['Day 1: Acclimatize, visit Tawang Monastery.', 'Day 2: Visit the Tawang War Memorial.', 'Day 3: Explore Pankang Teng Tso Lake.'],
  '5-day': ['Day 1: Acclimatize, visit Tawang Monastery.', 'Day 2: Visit the Tawang War Memorial.', 'Day 3: Explore Pankang Teng Tso Lake.', 'Day 4: Day trip to Bum La Pass (permit required).', 'Day 5: Visit the Jaswant Garh war memorial on your way back.'],
  '7-day': ['Day 1: Acclimatize, visit Tawang Monastery.', 'Day 2: Visit the Tawang War Memorial.', 'Day 3: Explore Pankang Teng Tso Lake.', 'Day 4: Day trip to Bum La Pass (permit required).', 'Day 5: Visit the Jaswant Garh war memorial.', 'Day 6: Visit Nuranang Falls.', 'Day 7: Explore the local markets.']}},
'Auli': {'budget': 40, 'climate': 10, 'interest': 90, 'weather_city': 'Auli', 'itinerary': {
  '3-day': ['Day 1: Arrive in Joshimath, take ropeway to Auli.', 'Day 2: Enjoy skiing on the slopes.', 'Day 3: Visit Auli Artificial Lake.'],
  '5-day': ['Day 1: Arrive in Joshimath, take ropeway to Auli.', 'Day 2: Enjoy skiing on the slopes.', 'Day 3: Visit Auli Artificial Lake.', 'Day 4: Trek to Gurso Bugyal meadow.', 'Day 5: Take the chairlift for panoramic views.'],
  '7-day': ['Day 1: Arrive in Joshimath, take ropeway to Auli.', 'Day 2: Enjoy skiing on the slopes.', 'Day 3: Visit Auli Artificial Lake.', 'Day 4: Trek to Gurso Bugyal meadow.', 'Day 5: Take the chairlift for panoramic views.', 'Day 6: Explore Joshimath town.', 'Day 7: Final views of Nanda Devi peak and depart.']}},
'Mount Abu': {'budget': 45, 'climate': 50, 'interest': 60, 'weather_city': 'Mount Abu', 'itinerary': {
  '3-day': ['Day 1: Boat on Nakki Lake.', 'Day 2: Visit the Dilwara Jain Temples.', 'Day 3: Go to Guru Shikhar, the highest point.'],
  '5-day': ['Day 1: Boat on Nakki Lake.', 'Day 2: Visit the Dilwara Jain Temples.', 'Day 3: Go to Guru Shikhar, the highest point.', 'Day 4: Visit the Achalgarh Fort.', 'Day 5: Explore the Toad Rock and Sunset Point.'],
  '7-day': ['Day 1: Boat on Nakki Lake.', 'Day 2: Visit the Dilwara Jain Temples.', 'Day 3: Go to Guru Shikhar, the highest point.', 'Day 4: Visit the Achalgarh Fort.', 'Day 5: Explore the Toad Rock and Sunset Point.', 'Day 6: Visit the Peace Park.', 'Day 7: Relax and explore local markets.']}},
'Mahabaleshwar': {'budget': 40, 'climate': 60, 'interest': 70, 'weather_city': 'Mahabaleshwar', 'itinerary': {
  '3-day': ['Day 1: Arrive, boat on Venna Lake.', "Day 2: Visit viewpoints like Arthur's Seat and Kate's Point.", 'Day 3: Explore a strawberry farm (seasonal).'],
  '5-day': ['Day 1: Arrive, boat on Venna Lake.', "Day 2: Visit viewpoints like Arthur's Seat and Kate's Point.", 'Day 3: Explore a strawberry farm (seasonal).', 'Day 4: Visit the ancient Mahabaleshwar Temple.', 'Day 5: Day trip to Panchgani and Table Land.'],
  '7-day': ['Day 1: Arrive, boat on Venna Lake.', "Day 2: Visit viewpoints like Arthur's Seat and Kate's Point.", 'Day 3: Explore a strawberry farm (seasonal).', 'Day 4: Visit the ancient Mahabaleshwar Temple.', 'Day 5: Day trip to Panchgani and Table Land.', 'Day 6: Visit Lingmala Waterfall.', 'Day 7: Explore the local market.']}},
'Lonavala': {'budget': 35, 'climate': 65, 'interest': 55, 'weather_city': 'Lonavala', 'itinerary': {
  '3-day': ['Day 1: Visit Bhushi Dam.', 'Day 2: Explore Karla and Bhaja Caves.', 'Day 3: Visit Tiger Point for views.'],
  '5-day': ['Day 1: Visit Bhushi Dam.', 'Day 2: Explore Karla and Bhaja Caves.', 'Day 3: Visit Tiger Point for views.', 'Day 4: Visit Lohagad Fort.', 'Day 5: Relax at Pawna Lake.'],
  '7-day': ['Day 1: Visit Bhushi Dam.', 'Day 2: Explore Karla and Bhaja Caves.', 'Day 3: Visit Tiger Point for views.', 'Day 4: Visit Lohagad Fort.', 'Day 5: Relax at Pawna Lake.', 'Day 6: Explore the town and try the famous chikki.', 'Day 7: Visit the Della Adventure Park.']}},
'Pune': {'budget': 50, 'climate': 70, 'interest': 50, 'weather_city': 'Pune', 'itinerary': {
  '3-day': ['Day 1: Visit Shaniwar Wada.', 'Day 2: Explore the Aga Khan Palace.', 'Day 3: Walk through Koregaon Park.'],
  '5-day': ['Day 1: Visit Shaniwar Wada.', 'Day 2: Explore the Aga Khan Palace.', 'Day 3: Walk through Koregaon Park.', 'Day 4: Visit Dagdusheth Halwai Ganpati Temple.', 'Day 5: Day trip to Sinhagad Fort.'],
  '7-day': ['Day 1: Visit Shaniwar Wada.', 'Day 2: Explore the Aga Khan Palace.', 'Day 3: Walk through Koregaon Park.', 'Day 4: Visit Dagdusheth Halwai Ganpati Temple.', 'Day 5: Day trip to Sinhagad Fort.', 'Day 6: Explore the Raja Dinkar Kelkar Museum.', 'Day 7: Enjoy the Osho Garden.']}},
'Mumbai': {'budget': 60, 'climate': 85, 'interest': 55, 'weather_city': 'Mumbai', 'itinerary': {
  '3-day': ['Day 1: Visit the Gateway of India and take a ferry to Elephanta Caves.', 'Day 2: Explore Marine Drive and Juhu Beach.', 'Day 3: Experience local trains and visit Colaba Causeway.'],
  '5-day': ['Day 1: Visit the Gateway of India and take a ferry to Elephanta Caves.', 'Day 2: Explore Marine Drive and Juhu Beach.', 'Day 3: Experience local trains and visit Colaba Causeway.', 'Day 4: Visit Haji Ali Dargah and Siddhivinayak Temple.', 'Day 5: Explore the film city or a Bollywood tour.'],
  '7-day': ['Day 1: Visit the Gateway of India and take a ferry to Elephanta Caves.', 'Day 2: Explore Marine Drive and Juhu Beach.', 'Day 3: Experience local trains and visit Colaba Causeway.', 'Day 4: Visit Haji Ali Dargah and Siddhivinayak Temple.', 'Day 5: Explore the film city or a Bollywood tour.', 'Day 6: Explore the Bandra-Worli Sea Link and Bandstand.', 'Day 7: Food tour from street food to fine dining.']}},
'Delhi': {'budget': 55, 'climate': 80, 'interest': 60, 'weather_city': 'Delhi', 'itinerary': {
  '3-day': ['Day 1: Explore Old Delhi (Red Fort, Chandni Chowk).', 'Day 2: Visit New Delhi (India Gate, Qutub Minar).', "Day 3: Explore Humayun's Tomb and Lotus Temple."],
  '5-day': ['Day 1: Explore Old Delhi (Red Fort, Chandni Chowk).', 'Day 2: Visit New Delhi (India Gate, Qutub Minar).', "Day 3: Explore Humayun's Tomb and Lotus Temple.", 'Day 4: Visit Lodhi Garden and Khan Market.', 'Day 5: Explore the Akshardham Temple.'],
  '7-day': ['Day 1: Explore Old Delhi (Red Fort, Chandni Chowk).', 'Day 2: Visit New Delhi (India Gate, Qutub Minar).', "Day 3: Explore Humayun's Tomb and Lotus Temple.", 'Day 4: Visit Lodhi Garden and Khan Market.', 'Day 5: Explore the Akshardham Temple.', 'Day 6: Visit the National Museum.', 'Day 7: Explore Hauz Khas Village.']}},
'Chandigarh': {'budget': 45, 'climate': 65, 'interest': 50, 'weather_city': 'Chandigarh', 'itinerary': {
  '3-day': ['Day 1: Boat on Sukhna Lake.', 'Day 2: Explore the Rock Garden.', 'Day 3: Visit the Rose Garden.'],
  '5-day': ['Day 1: Boat on Sukhna Lake.', 'Day 2: Explore the Rock Garden.', 'Day 3: Visit the Rose Garden.', "Day 4: Explore the Capitol Complex (Le Corbusier's work).", 'Day 5: Shopping in Sector 17.'],
  '7-day': ['Day 1: Boat on Sukhna Lake.', 'Day 2: Explore the Rock Garden.', 'Day 3: Visit the Rose Garden.', "Day 4: Explore the Capitol Complex (Le Corbusier's work).", 'Day 5: Shopping in Sector 17.', 'Day 6: Day trip to Pinjore Gardens.', "Day 7: Relax and explore the city's architecture."]}},
'Bangalore': {'budget': 60, 'climate': 70, 'interest': 55, 'weather_city': 'Bangalore', 'itinerary': {
  '3-day': ['Day 1: Visit Lalbagh Botanical Garden.', 'Day 2: Explore Bangalore Palace.', 'Day 3: Stroll through Cubbon Park.'],
  '5-day': ['Day 1: Visit Lalbagh Botanical Garden.', 'Day 2: Explore Bangalore Palace.', 'Day 3: Stroll through Cubbon Park.', 'Day 4: Visit the ISKCON Temple.', 'Day 5: Experience the nightlife and breweries in Indiranagar.'],
  '7-day': ['Day 1: Visit Lalbagh Botanical Garden.', 'Day 2: Explore Bangalore Palace.', 'Day 3: Stroll through Cubbon Park.', 'Day 4: Visit the ISKCON Temple.', 'Day 5: Experience the nightlife and breweries in Indiranagar.', 'Day 6: Day trip to Nandi Hills for sunrise.', 'Day 7: Explore the Visvesvaraya Industrial & Technological Museum.']}},
'Hyderabad': {'budget': 55, 'climate': 75, 'interest': 60, 'weather_city': 'Hyderabad', 'itinerary': {
  '3-day': ['Day 1: Visit the Charminar and Laad Bazaar.', 'Day 2: Explore Golconda Fort.', 'Day 3: Tour the Ramoji Film City.'],
  '5-day': ['Day 1: Visit the Charminar and Laad Bazaar.', 'Day 2: Explore Golconda Fort.', 'Day 3: Tour the Ramoji Film City.', 'Day 4: Visit the Salar Jung Museum.', 'Day 5: Relax at Hussain Sagar Lake.'],
  '7-day': ['Day 1: Visit the Charminar and Laad Bazaar.', 'Day 2: Explore Golconda Fort.', 'Day 3: Tour the Ramoji Film City.', 'Day 4: Visit the Salar Jung Museum.', 'Day 5: Relax at Hussain Sagar Lake.', 'Day 6: Explore the Chowmahalla Palace.', 'Day 7: Enjoy Hyderabadi Biryani at a famous restaurant.']}},
'Chennai': {'budget': 60, 'climate': 90, 'interest': 55, 'weather_city': 'Chennai', 'itinerary': {
  '3-day': ['Day 1: Relax at Marina Beach.', 'Day 2: Visit the Kapaleeshwarar Temple.', 'Day 3: Explore Fort St. George.'],
  '5-day': ['Day 1: Relax at Marina Beach.', 'Day 2: Visit the Kapaleeshwarar Temple.', 'Day 3: Explore Fort St. George.', 'Day 4: Day trip to Mahabalipuram.', 'Day 5: Visit the Government Museum.'],
  '7-day': ['Day 1: Relax at Marina Beach.', 'Day 2: Visit the Kapaleeshwarar Temple.', 'Day 3: Explore Fort St. George.', 'Day 4: Day trip to Mahabalipuram.', 'Day 5: Visit the Government Museum.', 'Day 6: Explore the Kalakshetra Foundation.', 'Day 7: Enjoy South Indian cuisine.']}},
'Andaman': {'budget': 70, 'climate': 85, 'interest': 80, 'weather_city': 'Port Blair', 'itinerary': {
  '3-day': ['Day 1: Arrive in Port Blair, visit Cellular Jail.', 'Day 2: Ferry to Havelock Island (Swaraj Dweep), visit Radhanagar Beach.', 'Day 3: Return to Port Blair and depart.'],
  '5-day': ['Day 1: Arrive in Port Blair, visit Cellular Jail.', 'Day 2: Ferry to Havelock Island (Swaraj Dweep), visit Radhanagar Beach.', 'Day 3: Return to Port Blair and depart.', 'Day 4: Snorkeling or Scuba diving at Elephant Beach.', 'Day 5: Ferry to Neil Island (Shaheed Dweep) for a quieter experience.'],
  '7-day': ['Day 1: Arrive in Port Blair, visit Cellular Jail.', 'Day 2: Ferry to Havelock Island (Swaraj Dweep), visit Radhanagar Beach.', 'Day 3: Snorkeling or Scuba diving at Elephant Beach.', 'Day 4: Ferry to Neil Island (Shaheed Dweep) for a quieter experience.', 'Day 5: Return to Port Blair, explore Corbyns Cove Beach.', 'Day 6: Explore Baratang Island (limestone caves, mud volcano).', 'Day 7: Relax and shop for souvenirs in Port Blair.']}},
'Coorg': {'budget': 40, 'climate': 50, 'interest': 75, 'weather_city': 'Madikeri', 'itinerary': {
  '3-day': ["Day 1: Arrive in Madikeri, visit Raja's Seat.", 'Day 2: Explore a coffee plantation.', 'Day 3: Visit Abbey Falls.'],
  '5-day': ["Day 1: Arrive in Madikeri, visit Raja's Seat.", 'Day 2: Explore a coffee plantation.', 'Day 3: Visit Abbey Falls.', 'Day 4: Visit the Dubare Elephant Camp.', 'Day 5: Visit the Namdroling Monastery (Golden Temple).'],
  '7-day': ["Day 1: Arrive in Madikeri, visit Raja's Seat.", 'Day 2: Explore a coffee plantation.', 'Day 3: Visit Abbey Falls.', 'Day 4: Visit the Dubare Elephant Camp.', 'Day 5: Visit the Namdroling Monastery (Golden Temple).', 'Day 6: Trek to Tadiandamol Peak.', 'Day 7: Relax and enjoy the serene environment.']}},
'Nainital': {'budget': 35, 'climate': 40, 'interest': 60, 'weather_city': 'Nainital', 'itinerary': {
  '3-day': ['Day 1: Arrive, boat on Naini Lake.', 'Day 2: Take the cable car to Snow View Point.', 'Day 3: Stroll on the Mall Road.'],
  '5-day': ['Day 1: Arrive, boat on Naini Lake.', 'Day 2: Take the cable car to Snow View Point.', 'Day 3: Stroll on the Mall Road.', 'Day 4: Visit the Nainital Zoo.', 'Day 5: Day trip to Bhimtal and Sattal lakes.'],
  '7-day': ['Day 1: Arrive, boat on Naini Lake.', 'Day 2: Take the cable car to Snow View Point.', 'Day 3: Stroll on the Mall Road.', 'Day 4: Visit the Nainital Zoo.', 'Day 5: Day trip to Bhimtal and Sattal lakes.', 'Day 6: Trek to Tiffin Top for views.', 'Day 7: Visit the Naina Devi Temple.']}},
'Mussoorie': {'budget': 40, 'climate': 35, 'interest': 65, 'weather_city': 'Mussoorie', 'itinerary': {
  '3-day': ['Day 1: Arrive, walk on the Mall Road.', 'Day 2: Visit Kempty Falls.', 'Day 3: Take the cable car to Gun Hill.'],
  '5-day': ['Day 1: Arrive, walk on the Mall Road.', 'Day 2: Visit Kempty Falls.', 'Day 3: Take the cable car to Gun Hill.', "Day 4: Visit Landour and St. Paul's Church.", 'Day 5: Day trip to Dhanaulti.'],
  '7-day': ['Day 1: Arrive, walk on the Mall Road.', 'Day 2: Visit Kempty Falls.', 'Day 3: Take the cable car to Gun Hill.', "Day 4: Visit Landour and St. Paul's Church.", 'Day 5: Day trip to Dhanaulti.', "Day 6: Trek to George Everest's House.", 'Day 7: Relax at a cafe with a view.']}},
'Bikaner': {'budget': 35, 'climate': 90, 'interest': 60, 'weather_city': 'Bikaner', 'itinerary': {
  '3-day': ['Day 1: Visit Junagarh Fort.', 'Day 2: Visit the Karni Mata Temple (Rat Temple).', 'Day 3: Explore the old city and havelis.'],
  '5-day': ['Day 1: Visit Junagarh Fort.', 'Day 2: Visit the Karni Mata Temple (Rat Temple).', 'Day 3: Explore the old city and havelis.', 'Day 4: Visit the National Research Centre on Camel.', 'Day 5: Take a short camel safari.'],
  '7-day': ['Day 1: Visit Junagarh Fort.', 'Day 2: Visit the Karni Mata Temple (Rat Temple).', 'Day 3: Explore the old city and havelis.', 'Day 4: Visit the National Research Centre on Camel.', 'Day 5: Take a short camel safari.', 'Day 6: Visit the Lalgarh Palace.', 'Day 7: Try local Bikaneri snacks.']}},
'Jaisalmer': {'budget': 40, 'climate': 95, 'interest': 70, 'weather_city': 'Jaisalmer', 'itinerary': {
  '3-day': ['Day 1: Explore Jaisalmer Fort.', 'Day 2: Overnight desert safari in the Sam Sand Dunes.', 'Day 3: Visit Patwon Ki Haveli.'],
  '5-day': ['Day 1: Explore Jaisalmer Fort.', 'Day 2: Overnight desert safari in the Sam Sand Dunes.', 'Day 3: Visit Patwon Ki Haveli.', 'Day 4: Visit Gadisar Lake.', 'Day 5: Explore the abandoned village of Kuldhara.'],
  '7-day': ['Day 1: Explore Jaisalmer Fort.', 'Day 2: Overnight desert safari in the Sam Sand Dunes.', 'Day 3: Visit Patwon Ki Haveli.', 'Day 4: Visit Gadisar Lake.', 'Day 5: Explore the abandoned village of Kuldhara.', 'Day 6: Visit Bada Bagh (royal cenotaphs).', 'Day 7: Shop for local crafts.']}},
'Puri': {'budget': 35, 'climate': 85, 'interest': 60, 'weather_city': 'Puri', 'itinerary': {
  '3-day': ['Day 1: Visit Jagannath Temple.', 'Day 2: Relax at Puri Beach.', 'Day 3: Day trip to Konark Sun Temple.'],
  '5-day': ['Day 1: Visit Jagannath Temple.', 'Day 2: Relax at Puri Beach.', 'Day 3: Day trip to Konark Sun Temple.', 'Day 4: Day trip to Chilika Lake for dolphin watching.', 'Day 5: Explore the Raghurajpur heritage crafts village.'],
  '7-day': ['Day 1: Visit Jagannath Temple.', 'Day 2: Relax at Puri Beach.', 'Day 3: Day trip to Konark Sun Temple.', 'Day 4: Day trip to Chilika Lake for dolphin watching.', 'Day 5: Explore the Raghurajpur heritage crafts village.', 'Day 6: Visit the Sunara Gauranga Temple.', 'Day 7: Relax and enjoy local seafood.']}},
'Konark': {'budget': 35, 'climate': 80, 'interest': 70, 'weather_city': 'Konark', 'itinerary': {
  '3-day': ['Day 1: Explore the Konark Sun Temple.', 'Day 2: Relax at Chandrabhaga Beach.', 'Day 3: Visit the Archaeological Survey of India Museum.'],
  '5-day': ['Day 1: Explore the Konark Sun Temple.', 'Day 2: Relax at Chandrabhaga Beach.', 'Day 3: Visit the Archaeological Survey of India Museum.', 'Day 4: Day trip to Puri.', 'Day 5: Explore the local markets.'],
  '7-day': ['Day 1: Explore the Konark Sun Temple.', 'Day 2: Relax at Chandrabhaga Beach.', 'Day 3: Visit the Archaeological Survey of India Museum.', 'Day 4: Day trip to Puri.', 'Day 5: Explore the local markets.', 'Day 6: Revisit the Sun Temple at a different time of day.', 'Day 7: Relax before departure.']}},
'Sikkim': {'budget': 45, 'climate': 35, 'interest': 75, 'weather_city': 'Gangtok', 'itinerary': {
  '3-day': ['Day 1: Arrive in Gangtok, explore MG Marg.', 'Day 2: Day trip to Tsomgo Lake and Baba Mandir.', 'Day 3: Visit Rumtek Monastery.'],
  '5-day': ['Day 1: Arrive in Gangtok, explore MG Marg.', 'Day 2: Day trip to Tsomgo Lake and Baba Mandir.', 'Day 3: Visit Rumtek Monastery.', 'Day 4: Travel to Pelling in West Sikkim.', 'Day 5: Visit Pemayangtse Monastery and Khecheopalri Lake.'],
  '7-day': ['Day 1: Arrive in Gangtok, explore MG Marg.', 'Day 2: Day trip to Tsomgo Lake and Baba Mandir.', 'Day 3: Visit Rumtek Monastery.', 'Day 4: Travel to Pelling in West Sikkim.', 'Day 5: Visit Pemayangtse Monastery and Khecheopalri Lake.', 'Day 6: Explore Yuksom, the first capital of Sikkim.', 'Day 7: Travel back to Gangtok for departure.']}},
'Gangtok': {'budget': 45, 'climate': 40, 'interest': 65, 'weather_city': 'Gangtok', 'itinerary': {
  '3-day': ['Day 1: Arrive, explore MG Marg.', 'Day 2: Day trip to Tsomgo Lake and Baba Mandir.', 'Day 3: Visit Rumtek Monastery.'],
  '5-day': ['Day 1: Arrive, explore MG Marg.', 'Day 2: Day trip to Tsomgo Lake and Baba Mandir.', 'Day 3: Visit Rumtek Monastery.', 'Day 4: Visit the Namgyal Institute of Tibetology.', 'Day 5: Take the ropeway for city views.'],
  '7-day': ['Day 1: Arrive, explore MG Marg.', 'Day 2: Day trip to Tsomgo Lake and Baba Mandir.', 'Day 3: Visit Rumtek Monastery.', 'Day 4: Visit the Namgyal Institute of Tibetology.', 'Day 5: Take the ropeway for city views.', 'Day 6: Day trip to Nathu La Pass (permit required, closed Mon/Tues).', 'Day 7: Explore local viewpoints like Tashi View Point.']}},
'Ranthambore': {'budget': 50, 'climate': 75, 'interest': 90, 'weather_city': 'Sawai Madhopur', 'itinerary': {
  '3-day': ['Day 1: Arrive, afternoon jungle safari.', 'Day 2: Morning jungle safari, visit Ranthambore Fort.', 'Day 3: One more safari before departure.'],
  '5-day': ['Day 1: Arrive, afternoon jungle safari.', 'Day 2: Morning jungle safari, visit Ranthambore Fort.', 'Day 3: Another morning safari in a different zone.', 'Day 4: Visit the Rajiv Gandhi Regional Museum of Natural History.', 'Day 5: Explore the nearby villages.'],
  '7-day': ['Day 1: Arrive, afternoon jungle safari.', 'Day 2: Morning jungle safari, visit Ranthambore Fort.', 'Day 3: Another morning safari in a different zone.', 'Day 4: Visit the Rajiv Gandhi Regional Museum of Natural History.', 'Day 5: Explore the nearby villages.', "Day 6: Bird watching tour around the park's lakes.", 'Day 7: Relax at your resort.']}},
'Kaziranga': {'budget': 50, 'climate': 65, 'interest': 85, 'weather_city': 'Jorhat', 'itinerary': {
  '3-day': ['Day 1: Arrive, afternoon jeep safari.', 'Day 2: Early morning elephant safari, another jeep safari.', 'Day 3: Visit a tea garden before departure.'],
  '5-day': ['Day 1: Arrive, afternoon jeep safari.', 'Day 2: Early morning elephant safari, another jeep safari.', 'Day 3: Visit a tea garden.', 'Day 4: Explore a different safari zone (e.g., Eastern Range for birds).', 'Day 5: Visit a local Mising village.'],
  '7-day': ['Day 1: Arrive, afternoon jeep safari.', 'Day 2: Early morning elephant safari, another jeep safari.', 'Day 3: Visit a tea garden.', 'Day 4: Explore a different safari zone (e.g., Eastern Range for birds).', 'Day 5: Visit a local Mising village.', 'Day 6: Visit the Kaziranga Orchid and Biodiversity Park.', 'Day 7: Relax and enjoy the natural surroundings.']}},
'Jim Corbett': {'budget': 45, 'climate': 60, 'interest': 80, 'weather_city': 'Ramnagar', 'itinerary': {
  '3-day': ['Day 1: Arrive, afternoon safari in Jhirna/Dhela zone.', 'Day 2: Morning safari in Bijrani zone.', 'Day 3: Visit Garjia Temple before departure.'],
  '5-day': ['Day 1: Arrive, afternoon safari in Jhirna/Dhela zone.', 'Day 2: Morning safari in Bijrani zone.', 'Day 3: Visit Garjia Temple.', 'Day 4: Stay overnight in the Dhikala forest rest house (if booked in advance).', 'Day 5: Morning safari from Dhikala and depart.'],
  '7-day': ['Day 1: Arrive, afternoon safari in Jhirna/Dhela zone.', 'Day 2: Morning safari in Bijrani zone.', 'Day 3: Visit Garjia Temple.', 'Day 4: Stay overnight in the Dhikala forest rest house (if booked in advance).', 'Day 5: Morning safari from Dhikala.', 'Day 6: Visit the Corbett Waterfall.', 'Day 7: Explore the Corbett Museum at Kaladhungi.']}},
'Spiti Valley': {'budget': 50, 'climate': 15, 'interest': 90, 'weather_city': 'Kaza', 'itinerary': {
  '3-day': ['Day 1: Arrive in Kaza, acclimatize.', 'Day 2: Visit Key Monastery and Kibber village.', "Day 3: Visit the world's highest post office in Hikkim."],
  '5-day': ['Day 1: Arrive in Kaza, acclimatize.', 'Day 2: Visit Key Monastery and Kibber village.', "Day 3: Visit the world's highest post office in Hikkim.", 'Day 4: Visit Langza village (Buddha statue) and Komic village.', 'Day 5: Explore the Tabo Monastery.'],
  '7-day': ['Day 1: Arrive in Kaza, acclimatize.', 'Day 2: Visit Key Monastery and Kibber village.', "Day 3: Visit the world's highest post office in Hikkim.", 'Day 4: Visit Langza village (Buddha statue) and Komic village.', 'Day 5: Explore the Tabo Monastery.', 'Day 6: Visit Dhankar Monastery and Lake.', 'Day 7: Drive towards Chandratal Lake (seasonal).']}},
'Munnar': {'budget': 40, 'climate': 50, 'interest': 75, 'weather_city': 'Munnar', 'itinerary': {
  '3-day': ['Day 1: Arrive, visit a tea plantation.', 'Day 2: Visit Mattupetty Dam and Echo Point.', 'Day 3: Explore the Tea Museum.'],
  '5-day': ['Day 1: Arrive, visit a tea plantation.', 'Day 2: Visit Mattupetty Dam and Echo Point.', 'Day 3: Explore the Tea Museum.', 'Day 4: Visit Eravikulam National Park (for Nilgiri Tahr).', 'Day 5: Visit Top Station for views.'],
  '7-day': ['Day 1: Arrive, visit a tea plantation.', 'Day 2: Visit Mattupetty Dam and Echo Point.', 'Day 3: Explore the Tea Museum.', 'Day 4: Visit Eravikulam National Park (for Nilgiri Tahr).', 'Day 5: Visit Top Station for views.', "Day 6: Day trip to Kolukkumalai Tea Estate (world's highest).", 'Day 7: Relax and enjoy the scenery.']}},
'Yercaud': {'budget': 35, 'climate': 60, 'interest': 60, 'weather_city': 'Yercaud', 'itinerary': {
  '3-day': ['Day 1: Arrive, boat on Yercaud Lake (Emerald Lake).', "Day 2: Visit Lady's Seat viewpoint.", 'Day 3: Explore the Killiyur Falls.'],
  '5-day': ['Day 1: Arrive, boat on Yercaud Lake (Emerald Lake).', "Day 2: Visit Lady's Seat viewpoint.", 'Day 3: Explore the Killiyur Falls.', 'Day 4: Visit the Pagoda Point.', 'Day 5: Explore the Botanical Garden.'],
  '7-day': ['Day 1: Arrive, boat on Yercaud Lake (Emerald Lake).', "Day 2: Visit Lady's Seat viewpoint.", 'Day 3: Explore the Killiyur Falls.', 'Day 4: Visit the Pagoda Point.', 'Day 5: Explore the Botanical Garden.', 'Day 6: Visit the Servarayan Temple.', 'Day 7: Relax at a coffee estate.']}},
'Lavasa': {'budget': 45, 'climate': 65, 'interest': 50, 'weather_city': 'Lavasa', 'itinerary': {
  '3-day': ['Day 1: Arrive, walk along the lakeside promenade.', 'Day 2: Try some water sports on the lake.', 'Day 3: Relax at a cafe with a view.'],
  '5-day': ['Day 1: Arrive, walk along the lakeside promenade.', 'Day 2: Try some water sports on the lake.', 'Day 3: Relax at a cafe with a view.', 'Day 4: Go for a nature trail walk.', 'Day 5: Enjoy the planned city architecture.'],
  '7-day': ['Day 1: Arrive, walk along the lakeside promenade.', 'Day 2: Try some water sports on the lake.', 'Day 3: Relax at a cafe with a view.', 'Day 4: Go for a nature trail walk.', 'Day 5: Enjoy the planned city architecture.', 'Day 6: Relax and unwind.', 'Day 7: Enjoy the peaceful environment before departure.']}},
'Havelock Island': {'budget': 70, 'climate': 85, 'interest': 85, 'weather_city': 'Havelock', 'itinerary': {
  '3-day': ['Day 1: Arrive, visit Radhanagar Beach for sunset.', 'Day 2: Snorkeling or Scuba at Elephant Beach.', 'Day 3: Relax at Kalapathar Beach.'],
  '5-day': ['Day 1: Arrive, visit Radhanagar Beach for sunset.', 'Day 2: Snorkeling or Scuba at Elephant Beach.', 'Day 3: Relax at Kalapathar Beach.', 'Day 4: Try kayaking through the mangroves.', 'Day 5: Explore the local market and cafes.'],
  '7-day': ['Day 1: Arrive, visit Radhanagar Beach for sunset.', 'Day 2: Snorkeling or Scuba at Elephant Beach.', 'Day 3: Relax at Kalapathar Beach.', 'Day 4: Try kayaking through the mangroves.', 'Day 5: Explore the local market and cafes.', 'Day 6: Take a certified scuba diving course (PADI).', 'Day 7: Enjoy a final beach day before departing.']}},
'Rameswaram': {'budget': 40, 'climate': 90, 'interest': 70, 'weather_city': 'Rameswaram', 'itinerary': {
  '3-day': ['Day 1: Visit the Ramanathaswamy Temple.', 'Day 2: Visit Dhanushkodi (Ghost Town).', 'Day 3: See the Pamban Bridge.'],
  '5-day': ['Day 1: Visit the Ramanathaswamy Temple.', 'Day 2: Visit Dhanushkodi (Ghost Town).', 'Day 3: See the Pamban Bridge.', 'Day 4: Visit the APJ Abdul Kalam Memorial.', 'Day 5: Relax at Agnitheertham beach.'],
  '7-day': ['Day 1: Visit the Ramanathaswamy Temple.', 'Day 2: Visit Dhanushkodi (Ghost Town).', 'Day 3: See the Pamban Bridge.', 'Day 4: Visit the APJ Abdul Kalam Memorial.', 'Day 5: Relax at Agnitheertham beach.', 'Day 6: Explore the Five-faced Hanuman Temple.', 'Day 7: Revisit the main temple for a different ceremony.']}},
'Tirupati': {'budget': 30, 'climate': 85, 'interest': 65, 'weather_city': 'Tirupati', 'itinerary': {
  '3-day': ['Day 1: Arrive, prepare for temple visit.', 'Day 2: Darshan at Tirumala (Venkateswara Temple).', 'Day 3: Visit other local temples like Padmavathi Ammavari Temple.'],
  '5-day': ['Day 1: Arrive, prepare for temple visit.', 'Day 2: Darshan at Tirumala (Venkateswara Temple).', 'Day 3: Visit other local temples like Padmavathi Ammavari Temple.', 'Day 4: Visit the Sri Vari Museum.', 'Day 5: Explore the Talakona Waterfalls.'],
  '7-day': ['Day 1: Arrive, prepare for temple visit.', 'Day 2: Darshan at Tirumala (Venkateswara Temple).', 'Day 3: Visit other local temples like Padmavathi Ammavari Temple.', 'Day 4: Visit the Sri Vari Museum.', 'Day 5: Explore the Talakona Waterfalls.', 'Day 6: Visit the Deer Park.', 'Day 7: Relax before departure.']}},
'Varanasi': {'budget': 35, 'climate': 80, 'interest': 70, 'weather_city': 'Varanasi', 'itinerary': {
  '3-day': ['Day 1: Arrive, evening Ganga Aarti at Dashashwamedh Ghat.', 'Day 2: Morning boat ride on the Ganges, explore the ghats.', 'Day 3: Visit Sarnath, where Buddha gave his first sermon.'],
  '5-day': ['Day 1: Arrive, evening Ganga Aarti at Dashashwamedh Ghat.', 'Day 2: Morning boat ride on the Ganges, explore the ghats.', 'Day 3: Visit Sarnath, where Buddha gave his first sermon.', 'Day 4: Explore the narrow lanes and temples of the old city.', 'Day 5: Visit the Banaras Hindu University campus.'],
  '7-day': ['Day 1: Arrive, evening Ganga Aarti at Dashashwamedh Ghat.', 'Day 2: Morning boat ride on the Ganges, explore the ghats.', 'Day 3: Visit Sarnath, where Buddha gave his first sermon.', 'Day 4: Explore the narrow lanes and temples of the old city.', 'Day 5: Visit the Banaras Hindu University campus.', 'Day 6: Attend a classical music or yoga session.', 'Day 7: Food tour to try local delicacies like kachori and lassi.']}},
}
destinations = base_destinations.copy()
print(f"Loaded {len(base_destinations)} destinations")


## 4. Helper functions: scaling, fuzzy engine, feedback loop, weather
- **Scaling** maps INR budget (2,000-200,000) and climate (2-60 °C) onto the shared 0-100 universe of discourse.
- **`fuzzy_recommendation`** builds a triangular membership function (`trimf`, half-width 20) around each destination's three scores, reads the user's degree of membership with `interp_membership`, and sums the three values. The highest total wins (winner-take-all defuzzification).
- **`simulate_backpropagation`** nudges a rejected destination's scores by 5 points away from the user's preference for the rest of the session.

In [ ]:
# --- Scaling helpers: map real-world inputs onto the shared 0-100 "universe of discourse" ---
def map_budget_to_scale(budget_inr):
    min_budget, max_budget = 2000, 200000
    budget_inr = max(min_budget, min(budget_inr, max_budget))
    return ((budget_inr - min_budget) / (max_budget - min_budget)) * 100


def map_climate_to_scale(climate_celsius):
    min_climate, max_climate = 2, 60
    climate_celsius = max(min_climate, min(climate_celsius, max_climate))
    return ((climate_celsius - min_climate) / (max_climate - min_climate)) * 100


# --- Adaptive feedback: nudge a rejected destination away from the user's preferences ---
def simulate_backpropagation(disliked_place, user_prefs, state):
    nudge = 5
    current_destinations = state["destinations"]
    for attr, user_value in user_prefs.items():
        if attr in current_destinations[disliked_place]:
            place_value = current_destinations[disliked_place][attr]
            if user_value > 50 and place_value > 50:
                current_destinations[disliked_place][attr] = max(0, place_value - nudge)
            elif user_value < 50 and place_value < 50:
                current_destinations[disliked_place][attr] = min(100, place_value + nudge)
    state["destinations"] = current_destinations


# --- Core fuzzy engine: every destination is an implicit fuzzy rule ---
def fuzzy_recommendation(budget, climate, interest, excluded_places=[], current_destinations=None):
    if current_destinations is None:
        current_destinations = destinations
    scores = {}
    x_range = np.arange(0, 101, 1)
    for place, attrs in current_destinations.items():
        if place in excluded_places:
            continue
        b_score = fuzz.interp_membership(x_range, fuzz.trimf(x_range, [attrs['budget'] - 20, attrs['budget'], attrs['budget'] + 20]), budget)
        c_score = fuzz.interp_membership(x_range, fuzz.trimf(x_range, [attrs['climate'] - 20, attrs['climate'], attrs['climate'] + 20]), climate)
        i_score = fuzz.interp_membership(x_range, fuzz.trimf(x_range, [attrs['interest'] - 20, attrs['interest'], attrs['interest'] + 20]), interest)
        scores[place] = b_score + c_score + i_score
    if not scores:
        return None
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)[0][0]


# --- Live data: OpenWeatherMap ---
def get_weather(place):
    city_for_api = base_destinations.get(place, {}).get('weather_city', place)
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city_for_api}&appid={API_KEY}&units=metric"
    try:
        res = requests.get(url, timeout=5).json()
        if res.get("cod") != 200:
            return f" Weather info unavailable for {place} (API error: {res.get('message', 'Unknown')})."
        return f" Currently, it's **{res['main']['temp']}°C** with {res['weather'][0]['description']} in the {city_for_api} area."
    except requests.exceptions.RequestException:
        return " Network error."


def get_forecast(place):
    city_for_api = base_destinations.get(place, {}).get('weather_city', place)
    url = f"https://api.openweathermap.org/data/2.5/forecast?q={city_for_api}&appid={API_KEY}&units=metric"
    try:
        res = requests.get(url, timeout=5).json()
        if res.get("cod") != "200":
            return f" Forecast info unavailable for {place}."
        seen_days, forecast_lines = set(), []
        for item in res['list']:
            dt = datetime.datetime.fromtimestamp(item['dt'])
            date_str = dt.strftime("%A, %b %d")
            if date_str not in seen_days:
                forecast_lines.append(f"- **{date_str}:** {item['main']['temp']}°C, {item['weather'][0]['description']}")
                seen_days.add(date_str)
            if len(seen_days) >= 5:
                break
        return f" Here is the 5-day forecast for the **{city_for_api}** area:\n" + "\n".join(forecast_lines)
    except requests.exceptions.RequestException:
        return " Network error."


def get_stay_suggestions(place):
    query = quote_plus(f"hotels in {place}")
    return f""" Here are some links to find great stays in **{place}**:
- **Google Maps:** [Search on Google Maps](https://www.google.com/maps/search/?api=1&query={query})
- **Booking.com:** [Search on Booking.com](https://www.booking.com/searchresults.html?ss={quote_plus(place)})
- **Airbnb:** [Search on Airbnb](https://www.airbnb.com/s/{quote_plus(place)}/homes)"""


def display_itinerary(place, days):
    plan_key = f'{days}-day'
    if 'itinerary' in base_destinations[place] and plan_key in base_destinations[place]['itinerary']:
        plan = base_destinations[place]['itinerary'][plan_key]
        response = f" Here's a custom **{days}-Day Itinerary** for **{place}**:\n" + "\n".join([f"- **{day_plan}**" for day_plan in plan])
        return response
    return f"Sorry, I don't have a detailed {days}-day plan for {place} right now."


## 5. Chat logic (finite-state machine)
`start -> get_budget -> get_climate -> get_interest -> feedback -> explore`. Say `new` or `reset` at any point to start over.

In [ ]:
def init_state():
    return {"step": "start", "budget": None, "climate": None, "interest": None,
            "chosen_place": None, "disliked_places": [], "destinations": base_destinations.copy()}


def chat_response(message, history, state):
    """Finite-state machine: start -> get_budget -> get_climate -> get_interest -> feedback -> explore."""
    if state is None:
        state = init_state()
    step, bot_message = state["step"], ""
    options_text = "\n\nWhat's next? You can ask for `forecast`, `stays`, or an itinerary (`3-day`, `5-day`, `7-day`). You can also say `new` to start over or `exit` to end."

    if message.strip().lower() in ["new", "start over", "reset"]:
        state = init_state()
        bot_message = "Ok, let's start over! \n\nWhat is your approximate **budget** in INR (e.g., 50000)?"
        state["step"] = "get_budget"
        return bot_message, state

    if step == "start":
        bot_message = " Hi! I'm your Travel Assistant Chatbot.\n\nTo get started, what is your approximate **budget** in INR (e.g., 50000)?"
        state["step"] = "get_budget"

    elif step == "get_budget":
        try:
            val = int(message)
            state["budget"] = map_budget_to_scale(val)
            bot_message = f"Got it. A budget around ₹{val}.\n\nNow, what's your preferred **climate** in Celsius (e.g., 25)?"
            state["step"] = "get_climate"
        except ValueError:
            bot_message = "That doesn't look like a number. Please enter your budget in INR (e.g., 50000)."

    elif step == "get_climate":
        try:
            val = int(message)
            state["climate"] = map_climate_to_scale(val)
            bot_message = f"Perfect. A climate of {val}°C.\n\nLastly, on a scale of 0-100, how high is your interest in **adventure or heritage**?"
            state["step"] = "get_interest"
        except ValueError:
            bot_message = "That doesn't look like a number. Please enter your climate preference in Celsius (e.g., 25)."

    elif step == "get_interest":
        try:
            val = int(message)
            if 0 <= val <= 100:
                state["interest"] = val
                place = fuzzy_recommendation(state["budget"], state["climate"], state["interest"],
                                             state["disliked_places"], state["destinations"])
                if place:
                    state["chosen_place"] = place
                    bot_message = f"Based on your preferences, my recommendation is **{place}**!\n\n{get_weather(place)}\n\nDo you **like** this suggestion, or should I find **another** one?"
                    state["step"] = "feedback"
                else:
                    bot_message = "I couldn't find any more places. Would you like to **start over**?"
                    state["step"] = "start"
            else:
                bot_message = "Please enter a number between 0 and 100."
        except ValueError:
            bot_message = "That doesn't look like a number. Please enter a value between 0 and 100."

    elif step == "feedback":
        feedback = message.lower()
        if "like" in feedback or "yes" in feedback:
            bot_message = f"Great! What would you like to know about **{state['chosen_place']}**?" + options_text
            state["step"] = "explore"
        elif "another" in feedback or "no" in feedback:
            state["disliked_places"].append(state["chosen_place"])
            user_prefs = {'budget': state["budget"], 'climate': state["climate"], 'interest': state["interest"]}
            simulate_backpropagation(state["chosen_place"], user_prefs, state)
            place = fuzzy_recommendation(state["budget"], state["climate"], state["interest"],
                                         state["disliked_places"], state["destinations"])
            if place:
                state["chosen_place"] = place
                bot_message = f"No problem. Let's try: **{place}**!\n\n{get_weather(place)}\n\nDo you **like** this suggestion, or want **another** one?"
                state["step"] = "feedback"
            else:
                bot_message = "I've run out of recommendations. Would you like to **start over**?"
                state["step"] = "start"
        else:
            bot_message = "Sorry, I didn't understand. Please tell me if you **like** the suggestion or want **another** one."

    elif step == "explore":
        query, place = message.lower(), state["chosen_place"]
        if "forecast" in query:
            bot_message = get_forecast(place) + options_text
        elif "stay" in query:
            bot_message = get_stay_suggestions(place) + options_text
        elif "3" in query:
            bot_message = display_itinerary(place, 3) + options_text
        elif "5" in query:
            bot_message = display_itinerary(place, 5) + options_text
        elif "7" in query:
            bot_message = display_itinerary(place, 7) + options_text
        elif "exit" in query:
            bot_message = "Wishing you a safe and wonderful journey! Feel free to talk to me again anytime. Goodbye! "
            state = init_state()
        else:
            bot_message = f"Sorry, I didn't catch that. For **{place}**, I can help with `forecast`, `stays`, or itineraries." + options_text
        time.sleep(1)

    return bot_message, state


## 6. Launch the Gradio chat interface
Running this cell prints a public `gradio.live` link you can open or share while the notebook is running.

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title="Travel Assistant") as demo:
    gr.Markdown("## Travel Assistant ChatBot")
    gr.Markdown("Your personal guide to discovering the perfect travel destination in India. Say 'new' or 'reset' at any time to start over.")
    chatbot_state = gr.State()
    gr.ChatInterface(
        fn=chat_response,
        additional_inputs=[chatbot_state],
        additional_outputs=[chatbot_state],
        textbox=gr.Textbox(placeholder="Your personalized journey across India starts here! Let's plan your trip.", autofocus=True),
        type="messages",
        title="Travel Chatbot",
    )

demo.launch(debug=True, share=True)
